# ROPE Benchmark

Two parts:
1. **Forecast + interpolation throughput** — 120-hour forecast, 10 000 random point queries.
2. **Orbit integration** — 1-day ISS-like orbit under NRLMSIS-2.1 vs ROPE; altitude comparison.

Run from the `demo/` directory with the ROPE archive extracted alongside.

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone
from pathlib import Path
from scipy.integrate import solve_ivp

sys.path.insert(0, "../python")  # rope.py lives in python/ relative to the archive root

from rope import Rope
from demo_lib.perturbations import (
    perturbations_msis,
    drag_acceleration, gravity_perturbations,
    _gmst_rad,
    _MU, _RE,
)
from demo_lib.space_weather import SpaceWeather, SpaceWeatherEngine

CONF  = Path("rope.conf")
EPOCH = datetime(2020, 1, 1, tzinfo=timezone.utc)
BSTAR = 1e-2

_BLUE   = "#1565C0"
_ORANGE = "#E65100"
_PURPLE = "#4A148C"
_GRAY   = "#9E9E9E"

plt.rcParams.update({
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "font.size": 12,
})
print("imports ok")

---
## Part 1 — Forecast + Interpolation Throughput

In [ ]:
HORIZON = 25  # hours + 1 to forecast

rp = Rope(config_path=CONF)

t0 = time.perf_counter()
resp = rp.forecast(EPOCH, horizon=HORIZON)
t_forecast = time.perf_counter() - t0

print(f"Forecast wall time : {t_forecast:.2f} s")
print(f"Window start       : {resp['window_start']}")
print(f"Window end         : {resp['window_end']}")

In [ ]:
N   = 10_000
rng = np.random.default_rng(42)

t_offsets   = rng.uniform(0.0, (HORIZON - 1) * 3600.0, N)  # seconds into forecast window
query_times = [EPOCH + timedelta(seconds=float(s)) for s in t_offsets]
query_lsts  = rng.uniform(0.0,   24.0, N).tolist()
query_lats  = rng.uniform(-87.5, 87.5, N).tolist()
query_alts  = rng.uniform(100.0, 980.0, N).tolist()

with rp:
    t0 = time.perf_counter()
    batch_results = rp.get_batch(query_times, query_lsts, query_lats, query_alts)
    t_batch = time.perf_counter() - t0

densities     = np.array([r["density"]     for r in batch_results])
uncertainties = np.array([r["uncertainty"] for r in batch_results])

print(f"Batch query ({N:,} points)")
print(f"  Total time   : {t_batch * 1e3:.1f} ms")
print(f"  Per point    : {t_batch / N * 1e6:.2f} µs")
print(f"  Throughput   : {N / t_batch:,.0f} queries/s")

In [ ]:
# Per-point latency via individual get() calls
M = 500
single_times_us = []
with rp:
    for i in range(M):
        t0 = time.perf_counter()
        rp.get(query_times[i], query_lsts[i], query_lats[i], query_alts[i])
        single_times_us.append((time.perf_counter() - t0) * 1e6)

st = np.array(single_times_us)
print(f"Single-point query timing ({M} calls):")
print(f"  mean   : {st.mean():.1f} µs")
print(f"  median : {np.median(st):.1f} µs")
print(f"  p95    : {np.percentile(st, 95):.1f} µs")
print(f"  min    : {st.min():.1f} µs")
print(f"  max    : {st.max():.1f} µs")

In [ ]:
log_d = np.log10(densities)

print("Density statistics across 10 000 random points:")
print(f"  mean log10(ρ)  : {log_d.mean():.3f}")
print(f"  std  log10(ρ)  : {log_d.std():.3f}")
for p in (5, 25, 50, 75, 95):
    print(f"  p{p:<2}  ρ [kg/m³] : {np.percentile(densities, p):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(log_d, bins=60, color=_BLUE, edgecolor="none", alpha=0.85)
axes[0].set_xlabel(r"$\log_{10}\,\rho$ [kg m$^{-3}$]")
axes[0].set_ylabel("Count")
axes[0].set_title("Density distribution (10 000 random points)")
axes[0].grid(True, lw=0.4, color=_GRAY, alpha=0.5)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].hist(st, bins=40, color=_ORANGE, edgecolor="none", alpha=0.85)
axes[1].axvline(st.mean(), color="k", lw=1.5, ls="--",
                label=f"mean {st.mean():.1f} µs")
axes[1].set_xlabel("Single-query latency [µs]")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Single-point timing ({M} calls)")
axes[1].legend(frameon=False)
axes[1].grid(True, lw=0.4, color=_GRAY, alpha=0.5)
axes[1].spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

---
## Part 2 — Orbit Integration: ROPE vs NRLMSIS-2.1

ISS-like initial condition (alt ≈ 420 km, i = 51.6°). Integrated for 1 day.
The 120-hour ROPE forecast from Part 1 is reused — no second forecast call needed.

In [ ]:
def eom_msis(t, state, bstar, epoch, sw_engine, rho_log):
    r, v = state[:3], state[3:]
    a_grav = -_MU / np.linalg.norm(r)**3 * r
    current_date = epoch + timedelta(seconds=float(t))
    f107, f107a, ap7 = sw_engine.query(t)
    a_pert, rho = perturbations_msis(r, v, bstar,
                                     date=current_date,
                                     f107=f107, f107a=f107a, ap7=ap7)
    rho_log.append((t, rho))
    return np.concatenate([v, a_grav + a_pert])


def _gmst_coeffs(epoch):
    """Return (gmst_0, a1, a2) for delta_gmst(t) = (a1 + a2*t)*t [rad]."""
    gmst_0 = _gmst_rad(epoch)
    y, mo, d = epoch.year, epoch.month, epoch.day
    if mo <= 2:
        y -= 1; mo += 12
    A   = y // 100
    B   = 2 - A + A // 4
    jd0 = int(365.25*(y+4716)) + int(30.6001*(mo+1)) + d + epoch.hour/24.0 + B - 1524.5
    D0  = jd0 - 2451545.0
    C   = np.pi/180.0 * 0.000387933 / 36525.0**2
    a1  = np.pi/180.0 * 360.98564736629 / 86400.0 + C * 2.0 * D0 / 86400.0
    a2  = C / 86400.0**2
    return gmst_0, a1, a2


def make_eom_rope(epoch_ts, gmst_0, gmst_a1, gmst_a2, rp, rho_log):
    def _eom(t, state):
        r   = state[:3]
        v   = state[3:]
        r_n = np.linalg.norm(r)
        a_grav = -_MU / r_n**3 * r

        gmst    = gmst_0 + (gmst_a1 + gmst_a2 * t) * t
        lon_deg = np.degrees(np.arctan2(r[1], r[0]) - gmst) % 360.0
        if lon_deg > 180.0:
            lon_deg -= 360.0
        lat_deg = np.degrees(np.arcsin(np.clip(r[2] / r_n, -1.0, 1.0)))
        alt_km  = r_n - _RE
        lst     = (lon_deg / 15.0 + ((epoch_ts + t) % 86400.0) / 3600.0) % 24.0

        rho = rp.get_density(epoch_ts + t, lst, lat_deg, alt_km)
        rho_log.append((t, rho))

        a_pert = drag_acceleration(r, v, BSTAR, rho) + gravity_perturbations(r)
        out = np.empty(6)
        out[:3] = v
        out[3:] = a_grav + a_pert
        return out

    return _eom


def rv_to_elements(r, v):
    r_n    = np.linalg.norm(r)
    energy = 0.5 * np.linalg.norm(v)**2 - _MU / r_n
    a      = -_MU / (2.0 * energy)
    h      = np.cross(r, v)
    e      = np.linalg.norm(np.cross(v, h) / _MU - r / r_n)
    inc    = np.degrees(np.arccos(np.clip(h[2] / np.linalg.norm(h), -1, 1)))
    alt    = r_n - _RE
    return a, e, inc, alt


def alts_from_traj(r_hist, v_hist):
    return np.array([rv_to_elements(r_hist[k], v_hist[k])[3]
                     for k in range(len(r_hist))])

In [ ]:
DAYS = 1.0
DT   = 60.0  # output cadence, s

inc_rad = np.radians(51.6)
r0      = np.array([_RE + 420.0, 0.0, 0.0])
vc      = np.sqrt(_MU / np.linalg.norm(r0))
v0      = np.array([0.0, vc * np.cos(inc_rad), vc * np.sin(inc_rad)])
x0      = np.concatenate([r0, v0])

t_end  = DAYS * 86400.0
t_eval = np.arange(0.0, t_end + DT, DT)
ivp_kw = dict(method="LSODA", t_eval=t_eval, rtol=1e-9, atol=1e-9, dense_output=False)

print(f"IC  : alt={np.linalg.norm(r0) - _RE:.1f} km  i=51.6°  BStar={BSTAR:.2e}")
print(f"Span: {DAYS} day(s)  output step={DT:.0f} s")

print("\nLoading space weather ...")
sw        = SpaceWeather()
sw_engine = SpaceWeatherEngine(sw, EPOCH, DAYS)
print(sw.summary(EPOCH))

In [ ]:
print("Integrating MSIS-2.1 ...", flush=True)
rho_log_ms = []
t0 = time.perf_counter()
sol_ms = solve_ivp(eom_msis, (0.0, t_end), x0,
                   args=(BSTAR, EPOCH, sw_engine, rho_log_ms), **ivp_kw)
t_msis = time.perf_counter() - t0

assert sol_ms.success, f"MSIS failed: {sol_ms.message}"
print(f"  {sol_ms.nfev:,} evaluations  |  wall time: {t_msis:.1f} s")

In [ ]:
print("Integrating ROPE ...", flush=True)
rho_log_rope = []
epoch_ts                  = EPOCH.timestamp()
gmst_0, gmst_a1, gmst_a2 = _gmst_coeffs(EPOCH)

t0 = time.perf_counter()
with rp:
    eom_rope = make_eom_rope(epoch_ts, gmst_0, gmst_a1, gmst_a2, rp, rho_log_rope)
    sol_rope = solve_ivp(eom_rope, (0.0, t_end), x0, **ivp_kw)
t_rope = time.perf_counter() - t0

assert sol_rope.success, f"ROPE failed: {sol_rope.message}"
print(f"  {sol_rope.nfev:,} evaluations  |  wall time: {t_rope:.1f} s")

In [ ]:
print("Timing summary")
print(f"  Forecast (120 h)       : {t_forecast:.2f} s")
print(f"  Batch query (10 000)   : {t_batch * 1e3:.1f} ms  ({t_batch / N * 1e6:.2f} µs/point)")
print(f"  MSIS integration (1 d) : {t_msis:.1f} s  ({sol_ms.nfev:,} evals)")
print(f"  ROPE integration (1 d) : {t_rope:.1f} s  ({sol_rope.nfev:,} evals)")

In [ ]:
t_hr_ms   = sol_ms.t   / 3600.0
t_hr_rope = sol_rope.t / 3600.0

alt_ms   = alts_from_traj(sol_ms.y[:3].T,   sol_ms.y[3:].T)
alt_rope = alts_from_traj(sol_rope.y[:3].T, sol_rope.y[3:].T)

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(t_hr_ms,   alt_ms,   lw=1.2, color=_ORANGE, ls="--", label="MSIS-2.1")
ax.plot(t_hr_rope, alt_rope, lw=1.2, color=_PURPLE, ls=":",  label="ROPE")

ax.set_xlabel("Time [hr]")
ax.set_ylabel("Altitude [km]")
ax.set_title(
    f"ISS-like orbit — altitude vs time\n"
    f"epoch {EPOCH.date()}  BStar={BSTAR:.2e}  i=51.6°  alt\u2080=420 km"
)
ax.legend(frameon=False)
ax.grid(True, lw=0.4, color=_GRAY, alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

print(f"\nAltitude decay over {DAYS} day(s):")
print(f"  MSIS-2.1 : Δalt = {alt_ms[-1] - alt_ms[0]:.3f} km")
print(f"  ROPE     : Δalt = {alt_rope[-1] - alt_rope[0]:.3f} km")

In [ ]:
def _interp_log(log, t_eval):
    arr = np.array(log)
    return np.interp(t_eval, arr[:, 0], arr[:, 1])

t_hr    = t_eval / 3600.0
rho_ms  = _interp_log(rho_log_ms,   t_eval)
rho_rp  = _interp_log(rho_log_rope, t_eval)

fig, ax = plt.subplots(figsize=(10, 4))

ax.semilogy(t_hr, rho_ms, lw=1.0, color=_ORANGE, ls="--", label="MSIS-2.1")
ax.semilogy(t_hr, rho_rp, lw=1.0, color=_PURPLE, ls=":",  label="ROPE")

ax.set_xlabel("Time [hr]")
ax.set_ylabel(r"$\rho$ [kg m$^{-3}$]")
ax.set_title(
    f"Atmospheric density at satellite position\n"
    f"epoch {EPOCH.date()}  BStar={BSTAR:.2e}  i=51.6°  alt₀=420 km"
)
ax.legend(frameon=False)
ax.grid(True, lw=0.4, color=_GRAY, alpha=0.5, which="both")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
rp.shutdown()